# 긴 글·표·차량 안내도와 모델 입력 길이

이번 실험은 PDF 33~36쪽 차량 내부 안내도, 44쪽 표, 54쪽 안전벨트 설명을 사용합니다. 전체 PDF 자동 청킹의 완성본은 아닙니다.

**원문 묶음(부모 기록)**에는 한 주제의 설명과 그림을 함께 보관합니다. **검색 조각(자식 청크)**은 모델이 읽을 수 있는 짧은 글입니다. 검색 조각을 찾으면 연결된 원문 묶음을 가져와 완전한 설명과 경고문을 확인합니다.

이번에도 자료는 메모리에서 다룹니다. 청크 파일이나 Supabase 저장은 실행하지 않습니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 선택한 추가 모델의 실제 128토큰 한도로 부모 원문과 자식 검색 조각을 나눕니다.
- 추가: 안내도 번호 항목 보존, 표 관계 보존, 제목을 포함한 입력 길이 확인, 부모 ID·구간 기록.

전체 목록: [수업 기반·추가 적용 기록](../src/zzong_santafe_lag/learning_additions.md)


### 1. 모델이 실제로 읽을 수 있는 길이 확인

토큰은 모델이 글을 나눠 읽는 단위입니다. 글자 수와 같지 않습니다. 이미 다운로드한 모델 설정과 토크나이저(글을 토큰으로 나누는 도구)를 사용합니다. 아래 코드는 외부 추론 API를 호출하지 않습니다.

In [1]:
# [프로젝트 추가 기록] 선택한 추가 모델의 실제 128토큰 한도로 부모 원문과 자식 검색 조각을 나눕니다.
# [모델 추가] jhgan/ko-sroberta-multitask-mrl 선택 근거는 02번, 현재 설정은 src/zzong_santafe_lag/config.py에 있습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import re
from pypdf import PdfReader
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

# 이미 내려받은 파일만 사용해 모델 입력 길이를 확인합니다.
model_name = "jhgan/ko-sroberta-multitask-mrl"
config_path = hf_hub_download(model_name, "sentence_bert_config.json", local_files_only=True)
model_config = json.loads(Path(config_path).read_text(encoding="utf-8"))
token_budget = model_config["max_seq_length"]
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)

def token_count(text):
    """특수 토큰까지 포함한 실제 입력 길이를 셉니다. 글을 잘라 내지는 않습니다."""
    return len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
reader = PdfReader(str(project_folder / "data" / "santafe_hev_manual.pdf"))
page_texts = {n: reader.pages[n - 1].extract_text() or "" for n in [33, 34, 35, 36, 44, 54]}
print("모델 이름:", model_name)
print("모델 입력 한도:", token_budget, "토큰")

모델 이름: jhgan/ko-sroberta-multitask-mrl
모델 입력 한도: 128 토큰


### 2. 안내도와 표의 원문 묶음 만들기

차량 내부 I·II·III는 다음 페이지로 번호 목록이 이어집니다. 각 안내도의 제목을 경계로 잘라 다음 안내도와 섞이지 않게 합니다. 점선 뒤의 숫자는 안내도에 인쇄된 **매뉴얼 참조 쪽수**입니다. 실제 PDF 쪽수로 바꾸지는 않습니다.

44쪽 표는 확인한 두 이미지와 설명을 한 묶음으로 둡니다. 온도 눈금 범위를 임의로 차량의 사용 가능 온도라고 해석하지 않습니다.

In [2]:
# [프로젝트 추가 기록] 선택한 추가 모델의 실제 128토큰 한도로 부모 원문과 자식 검색 조각을 나눕니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def before_heading(text, heading):
    """확인한 다음 제목 앞까지만 원문을 가져옵니다."""
    marker = "\n" + heading + "\n"
    if marker not in text:
        raise ValueError(f"다음 제목을 찾지 못했습니다: {heading}")
    return text.split(marker, 1)[0]

def from_heading(text, heading):
    """확인한 제목부터 원문을 가져옵니다."""
    position = text.find(heading + "\n")
    if position < 0:
        raise ValueError(f"제목을 찾지 못했습니다: {heading}")
    return text[position:]

def image_parts(number, descriptions=None):
    """페이지와 이미지 이름을 함께 기록합니다. 모르는 그림은 설명을 만들지 않습니다."""
    descriptions = descriptions or {}
    parts = []
    for image in reader.pages[number - 1].images:
        path = project_folder / "data" / "zzong_santafe_lag" / "images" / f"pdf_page_{number}" / image.name
        parts.append({
            "name": image.name,
            "pdf_page_number": number,
            "size": image.image.size,
            "local_path": path.relative_to(project_folder).as_posix() if path.exists() else None,
            "description": descriptions.get(image.name),
        })
    return parts

# 앞서 실제 PDF의 번호 순서와 항목 수를 확인한 세 안내도만 처리합니다.
overview_specs = [
    ("차량 내부 I", 33, 28, [33, 34], 18,
     page_texts[33] + "\n" + before_heading(page_texts[34], "차량 내부 II")),
    ("차량 내부 II", 34, 29, [34, 35], 24,
     from_heading(page_texts[34], "차량 내부 II") + "\n" + before_heading(page_texts[35], "차량 내부 III")),
    ("차량 내부 III", 35, 30, [35, 36], 10,
     from_heading(page_texts[35], "차량 내부 III") + "\n" + page_texts[36]),
]
parent_records = []

for title, image_page, manual_page, pages, expected_count, raw_text in overview_specs:
    # '부품 이름 ... 참조 쪽수' 형태인 줄만 찾습니다.
    rows = re.findall(r"^([^\n]+?)\.{3,}\s*(\d+(?:\s*,\s*\d+)*)\s*$", raw_text, flags=re.MULTILINE)
    if len(rows) != expected_count:
        raise ValueError(f"{title}: 예상한 {expected_count}개 항목과 추출 결과가 다릅니다.")

    items = [
        {"number": i, "name": name.strip(), "target_manual_pages": [int(n) for n in pages_text.split(",")]}
        for i, (name, pages_text) in enumerate(rows, start=1)
    ]
    content_lines = [title, "본 도안은 적용된 사양에 따라 실제 차량과 다를 수 있습니다."]
    content_lines += [
        f'{item["number"]}: {item["name"]} — 매뉴얼 ' +
        ", ".join(str(n) + "쪽" for n in item["target_manual_pages"])
        for item in items
    ]
    parent_records.append({
        "content": "\n".join(content_lines),
        "raw_text": raw_text,
        "metadata": {
            "record_id": f"overview_{image_page}",
            "title": title,
            "pdf_page_number": image_page,
            "manual_page_number": manual_page,
            "source_pages": pages,
            "content_type": "vehicle_overview_navigation",
            "navigation_items": items,
            "number_mapping": "previously_confirmed_order",
        },
        "image_parts": image_parts(image_page, {"I1.jpg": title + "의 주요 부품 위치를 번호로 표시한 안내도."}),
    })

# 44쪽은 확인한 꼬리말만 검색 글에서 제외하며 raw_text에는 그대로 남깁니다.
table_raw = page_texts[44]
table_footer = "\n안내 및 차량 정보\n39"
if table_footer not in table_raw:
    raise ValueError("44쪽 표의 꼬리말을 다시 확인해야 합니다.")
table_descriptions = {
    "I1.jpg": "온도 눈금은 -30도부터 50도까지 표시되어 있다.",
    "I2.jpg": "1.6 T-GDi HEV 행에 0W-20이 표시되어 있다.",
}
parent_records.append({
    "content": table_raw.split(table_footer, 1)[0] + "\n[그림 설명] " + " ".join(table_descriptions.values()),
    "raw_text": table_raw,
    "metadata": {
        "record_id": "table_44",
        "title": "온도에 따른 엔진 오일 SAE 점도 분류표",
        "pdf_page_number": 44,
        "manual_page_number": 39,
        "source_pages": [44],
        "content_type": "table_with_images",
    },
    "image_parts": image_parts(44, table_descriptions),
})
for parent in parent_records:
    print(parent["metadata"]["title"], "| 원문 묶음 ID:", parent["metadata"]["record_id"],
          "| 항목 수:", len(parent["metadata"].get("navigation_items", [])),
          "| 그림:", len(parent["image_parts"]))

차량 내부 I | 원문 묶음 ID: overview_33 | 항목 수: 18 | 그림: 1
차량 내부 II | 원문 묶음 ID: overview_34 | 항목 수: 24 | 그림: 1
차량 내부 III | 원문 묶음 ID: overview_35 | 항목 수: 10 | 그림: 1
온도에 따른 엔진 오일 SAE 점도 분류표 | 원문 묶음 ID: table_44 | 항목 수: 0 | 그림: 2


### 3. 긴 설명과 경고문을 원문 묶음에 보관

54쪽 안전벨트 설명을 한 주제로 보관합니다. 경고 표시는 읽을 수 있는 글자로 바꾸고, 검색에 필요하지 않은 이미지 내부 코드는 제외합니다. 문장은 요약하거나 다시 쓰지 않습니다. 이 페이지의 그림은 설명을 아직 확인하지 않았으므로 페이지 단위 참조만 기록합니다.

In [3]:
# [프로젝트 추가 기록] 선택한 추가 모델의 실제 128토큰 한도로 부모 원문과 자식 검색 조각을 나눕니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def prepare_text(raw_text):
    """확인한 기호와 코드만 정리하고, 원문 문장은 유지합니다."""
    text = raw_text
    for old, new in {"҃Ҋ": "[경고]", "઱੄": "[주의]", "ӝ": "[참고]"}.items():
        text = text.replace(old, new)
    text = re.sub(r"^[12]C_[A-Za-z0-9_]+[ \t]*$", "", text, flags=re.MULTILINE)
    return re.sub(r"\s+", " ", text).strip()

long_raw = page_texts[54]
long_footer = "\n안전 및 주의 사항\n49"
if long_footer not in long_raw:
    raise ValueError("54쪽의 꼬리말을 다시 확인해야 합니다.")
parent_records.append({
    "content": prepare_text(long_raw.split(long_footer, 1)[0]),
    "raw_text": long_raw,
    "metadata": {
        "record_id": "seatbelt_54",
        "title": "안전벨트 착용",
        "pdf_page_number": 54,
        "manual_page_number": 49,
        "source_pages": [54],
        "content_type": "figure_with_text",
        "image_linkage_status": "page_level_needs_review",
    },
    "image_parts": image_parts(54),
})
for parent in parent_records:
    print(parent["metadata"]["title"], "| 글자:", len(parent["content"]),
          "| 토큰:", token_count(parent["content"]))

차량 내부 I | 글자: 553 | 토큰: 241
차량 내부 II | 글자: 729 | 토큰: 323
차량 내부 III | 글자: 299 | 토큰: 135
온도에 따른 엔진 오일 SAE 점도 분류표 | 글자: 131 | 토큰: 80
안전벨트 착용 | 글자: 894 | 토큰: 426


### 4. 모델 길이에 맞는 검색 조각 만들기

안내도는 번호 항목을 통째로, 표는 확인한 표 묶음을 통째로 사용합니다. 긴 글은 문장 경계를 우선 사용합니다. 한 문장이 모델 한도를 넘으면 공백 경계로 더 나누고 검토 대상으로 표시합니다.

짧은 검색 조각에는 원문 묶음의 ID를 기록합니다. 답변에 사용할 때는 연결된 원문 묶음 전체를 가져와야 합니다. 그림도 그 원문 묶음에 있으므로 청크를 나누면서 이미지 연결이 사라지지 않습니다.

In [4]:
# [프로젝트 추가 기록] 선택한 추가 모델의 실제 128토큰 한도로 부모 원문과 자식 검색 조각을 나눕니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def make_search_chunks(parent, budget):
    """유형별 경계를 지키며 실제 토큰 한도 안의 검색 조각을 만듭니다."""
    text = parent["content"]
    kind = parent["metadata"]["content_type"]
    prefix = "주제: " + parent["metadata"]["title"] + "\n"

    # 표는 한 묶음으로, 안내도는 한 줄의 번호 항목을 단위로 사용합니다.
    if kind == "table_with_images":
        units = [(0, len(text))]
    elif kind == "vehicle_overview_navigation":
        units = []
        start = 0
        for line in text.splitlines(keepends=True):
            units.append((start, start + len(line)))
            start += len(line)
    else:
        units = [(m.start(), m.end()) for m in re.finditer(r".+?(?:[.!?](?=\s|$)|$)", text, flags=re.DOTALL)]

    expanded_units = []
    split_for_budget = False
    for start, end in units:
        if token_count(prefix + text[start:end].strip()) <= budget:
            expanded_units.append((start, end))
            continue
        # 표의 수치나 안내도 번호 항목은 자동으로 잘라 관계를 깨뜨리지 않습니다.
        if kind in {"table_with_images", "vehicle_overview_navigation"}:
            raise ValueError(f'{parent["metadata"]["title"]}: 한 항목이 토큰 한도를 넘으므로 별도 검토가 필요합니다.')
        split_for_budget = True
        words = list(re.finditer(r"\S+\s*", text[start:end]))
        if not words:
            raise ValueError("분할할 문장을 확인해야 합니다.")
        # 선행 공백까지 포함해 원문 묶음의 검색 글을 빠짐없이 덮습니다.
        for i, word in enumerate(words):
            word_start = start if i == 0 else start + word.start()
            word_end = start + word.end()
            if token_count(prefix + text[word_start:word_end].strip()) > budget:
                raise ValueError("공백이 없는 긴 단어는 별도 검토가 필요합니다.")
            expanded_units.append((word_start, word_end))

    # 이웃한 단위를 합치되 모델 입력 한도는 넘지 않습니다.
    ranges = []
    group_start = group_end = None
    for start, end in expanded_units:
        if group_start is None:
            group_start, group_end = start, end
        elif token_count(prefix + text[group_start:end].strip()) <= budget:
            group_end = end
        else:
            ranges.append((group_start, group_end))
            group_start, group_end = start, end
    if group_start is not None:
        ranges.append((group_start, group_end))

    children = []
    for index, (start, end) in enumerate(ranges):
        children.append({
            "content": prefix + text[start:end].strip(),
            # 원문은 부모 주제 전체입니다. 구간 위치는 부모의 검색 글을 기준으로 기록합니다.
            "raw_text": parent["raw_text"],
            "metadata": {
                "record_id": parent["metadata"]["record_id"] + f"_chunk_{index}",
                "parent_record_id": parent["metadata"]["record_id"],
                "pdf_page_number": parent["metadata"]["pdf_page_number"],
                "manual_page_number": parent["metadata"]["manual_page_number"],
                "source_pages": parent["metadata"]["source_pages"],
                "content_type": kind,
                "chunk_index": index,
                "parent_content_start": start,
                "parent_content_end": end,
                "raw_text_scope": "parent_topic",
                "parent_read_required": True,
                "sentence_split_for_budget": split_for_budget,
                "token_count": token_count(prefix + text[start:end].strip()),
            },
            # 그림은 연결된 부모 기록에서 가져옵니다.
            "image_parts": [],
        })
    return children

search_chunks = []
parent_lookup = {parent["metadata"]["record_id"]: parent for parent in parent_records}
for parent in parent_records:
    children = make_search_chunks(parent, token_budget)
    search_chunks.extend(children)
    print(parent["metadata"]["title"], "| 검색 조각:", len(children),
          "| 최대 입력:", max(child["metadata"]["token_count"] for child in children), "토큰")

차량 내부 I | 검색 조각: 3 | 최대 입력: 128 토큰
차량 내부 II | 검색 조각: 3 | 최대 입력: 126 토큰
차량 내부 III | 검색 조각: 2 | 최대 입력: 117 토큰
온도에 따른 엔진 오일 SAE 점도 분류표 | 검색 조각: 1 | 최대 입력: 93 토큰
안전벨트 착용 | 검색 조각: 4 | 최대 입력: 127 토큰


### 5. 잘림·누락·원문 연결 확인

실제 토크나이저로 모든 검색 조각이 128토큰 안에 들어가는지 확인합니다. 자식 청크가 가리키는 구간이 부모의 검색 글을 빠짐없이 덮는지도 확인합니다. 숫자 목록은 앞서 확인한 18·24·10개가 유지돼야 합니다.

이 코드는 검색 결과에서 부모 기록을 가져오는 연결을 보여 줍니다. 아직 임베딩 유사도 검색을 수행하는 단계는 아닙니다.

In [5]:
# [프로젝트 추가 기록] 선택한 추가 모델의 실제 128토큰 한도로 부모 원문과 자식 검색 조각을 나눕니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
assert len(parent_records) == 5
assert [len(p["metadata"].get("navigation_items", [])) for p in parent_records[:3]] == [18, 24, 10]
assert len(parent_lookup["table_44"]["image_parts"]) == 2
assert all(token_count(chunk["content"]) <= token_budget for chunk in search_chunks)

# 검색용 부모 글에 빈 구간이나 중복 구간이 생기지 않았는지 확인합니다.
for parent_id, parent in parent_lookup.items():
    children = [c for c in search_chunks if c["metadata"]["parent_record_id"] == parent_id]
    cursor = 0
    for child in children:
        assert child["metadata"]["parent_content_start"] == cursor
        cursor = child["metadata"]["parent_content_end"]
    assert cursor == len(parent["content"])

# 긴 글에서 찾은 검색 조각을 통해 완전한 원문과 경고문을 가져옵니다.
seatbelt_child = next(c for c in search_chunks if c["metadata"]["parent_record_id"] == "seatbelt_54")
retrieved_parent = parent_lookup[seatbelt_child["metadata"]["parent_record_id"]]
assert "[경고]" in retrieved_parent["content"]
assert "안전벨트의 정상적인 착용 및 사용을 방해하는 물품들은 절대로 사용하지 마십시오." in retrieved_parent["content"]
assert "심각한 부상을 입을 수 있습니다." in retrieved_parent["content"]
assert "차단 클립" in retrieved_parent["content"] and "놀이방 매트" in retrieved_parent["content"]

print("원문 묶음:", len(parent_records))
print("검색 조각:", len(search_chunks))
print("모든 검색 조각의 모델 입력 한도:", "통과")
print("부모 검색 글의 구간 누락/중복:", "없음")
print("안내도 번호 항목:", "18 / 24 / 10 유지")
print("검색 조각에서 가져온 원문:", retrieved_parent["metadata"]["title"])
print("원문에 경고문과 주의사항:", "유지")
print("연결된 그림:", len(retrieved_parent["image_parts"]))
print("저장:", "메모리만 사용")

원문 묶음: 5
검색 조각: 13
모든 검색 조각의 모델 입력 한도: 통과
부모 검색 글의 구간 누락/중복: 없음
안내도 번호 항목: 18 / 24 / 10 유지
검색 조각에서 가져온 원문: 안전벨트 착용
원문에 경고문과 주의사항: 유지
연결된 그림: 1
저장: 메모리만 사용


이제 이 예제들은 실제 모델 입력 길이를 넘지 않는 검색 조각을 갖습니다. 나중에 답변 근거를 구성할 때는 부모 기록을 읽어 완전한 설명과 이미지를 제공합니다.

전체 문서 확장에는 아직 제목 확인과 표 구조 점검이 필요합니다. 예를 들어 43쪽 추천 오일 표의 추출 글에는 서로 다른 행의 사양이 붙어 있어 자동으로 행을 복원하지 않았습니다. 확인하지 않은 그림 설명도 추가하지 않았습니다.